In [1]:
import pandas as pd
import numpy as np
import warnings
from arch import arch_model
from statsmodels.stats.diagnostic import het_arch

# Ignore convergence warnings for cleaner console output
warnings.filterwarnings('ignore')

# EGARCH log-likelihoods are not globally concave, and on short regime
# sub-samples the optimiser can settle on a local optimum with economically
# implausible parameters (e.g. |alpha| in the tens, beta at zero). Earlier
# versions of this script reported a single fit per cell, which produced two
# such cells in the Recovery regime. We now fit each cell from several
# starting points and retain the highest-likelihood fit whose parameters lie
# in a plausible region.
N_STARTS = 8
PLAUSIBLE = dict(alpha_max=3.0, beta_lo=-0.05, beta_hi=1.02, gamma_max=1.5)


def is_plausible(params):
    return (abs(params.get('alpha[1]', 9)) <= PLAUSIBLE['alpha_max']
            and PLAUSIBLE['beta_lo'] <= params.get('beta[1]', 9) <= PLAUSIBLE['beta_hi']
            and abs(params.get('gamma[1]', 9)) <= PLAUSIBLE['gamma_max'])


def fit_egarch(y, seed=0):
    """Fit EGARCH(1,1)-t from multiple starts; return the best plausible fit."""
    rng = np.random.default_rng(seed)
    am = arch_model(y, vol='EGARCH', p=1, o=1, q=1, dist='t')
    candidates = []
    try:
        candidates.append(am.fit(disp='off', options={'maxiter': 2000}))
    except Exception:
        pass
    base = np.array([y.mean(), 0.05, 0.15, -0.05, 0.95, 8.0])
    for _ in range(N_STARTS):
        sv = base * (1 + rng.normal(0, 0.35, size=6))
        sv[2] = np.clip(abs(sv[2]), 0.02, 0.90)
        sv[4] = np.clip(sv[4], 0.30, 0.995)
        sv[5] = np.clip(abs(sv[5]), 4.0, 20.0)
        try:
            candidates.append(am.fit(disp='off', starting_values=sv,
                                     options={'maxiter': 2000}))
        except Exception:
            pass
    if not candidates:
        return None, False
    ok = [c for c in candidates if is_plausible(c.params)]
    pool = ok if ok else candidates
    return max(pool, key=lambda c: c.loglikelihood), bool(ok)


def main():
    print("Loading data...")
    df = pd.read_csv('../../master_dataset.csv', parse_dates=['date'], index_col='date')

    sectors = [
        'Banks', 'Capital_Goods', 'Consumer_Durables_and_Apparel',
        'Consumer_Services', 'Diversified_Financials', 'Energy',
        'Food,_Beverage_and_Tobacco', 'Materials', 'Real_Estate_Management_and_Development',
        'Retailing', 'Telecommunication_Services', 'Healthcare_Equipment_and_Services', 'Insurance'
    ]
    price_cols = ['ASPI'] + sectors

    print("Calculating log returns...")
    for col in price_cols:
        # x100 for percentage returns: required for the MLE to converge reliably.
        df[f'{col}_Return'] = np.log(df[col] / df[col].shift(1)) * 100

    df = df.dropna(subset=[f'{col}_Return' for col in price_cols])

    crisis_start, crisis_end = '2022-01-25', '2022-12-05'
    recovery_start, recovery_end = '2022-12-06', '2025-08-31'
    pre_crisis_end = (pd.to_datetime(crisis_start) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')

    periods = {
        'Pre-COVID': df.loc[:'2020-02-10'],
        'COVID': df.loc['2020-02-11':'2022-01-24'],
        'Crisis': df.loc[crisis_start:crisis_end],
        'Recovery': df.loc[recovery_start:recovery_end]
    }

    results = []
    print(f"Fitting EGARCH(1,1) with {N_STARTS + 1} starts per cell...")
    for period_name, period_data in periods.items():
        for col in price_cols:
            y = period_data[f'{col}_Return'].dropna()
            res, plausible = fit_egarch(y, seed=abs(hash((period_name, col))) % 10000)
            if res is None:
                print(f"Warning: all starts failed for {col} in {period_name}")
                continue

            p, se, pv = res.params, res.std_err, res.pvalues
            try:
                std_resid = (res.resid / res.conditional_volatility).dropna()
                arch_lm_pval = het_arch(std_resid, nlags=5)[1]
            except Exception:
                arch_lm_pval = np.nan

            results.append({
                'Period': period_name, 'Asset': col, 'N': len(y),
                'Omega': p.get('omega'), 'Omega SE': se.get('omega'),
                'Alpha (Shock)': p.get('alpha[1]'), 'Alpha SE': se.get('alpha[1]'),
                'Beta (Persistence)': p.get('beta[1]'), 'Beta SE': se.get('beta[1]'),
                'Gamma (Leverage)': p.get('gamma[1]'), 'Gamma SE': se.get('gamma[1]'),
                'Gamma P-Value': pv.get('gamma[1]'),
                'Nu (t d.o.f.)': p.get('nu'), 'Nu SE': se.get('nu'),
                'Log-Likelihood': res.loglikelihood, 'AIC': res.aic,
                'ARCH-LM P-Value': arch_lm_pval,
                'Significant Leverage?': 'Yes' if (pv.get('gamma[1]', 1) < 0.05 and p.get('gamma[1]', 0) < 0) else 'No',
                'Converged': res.convergence_flag == 0,
                'Plausible Region': plausible,
            })

    results_df = pd.DataFrame(results)
    results_df['Period'] = pd.Categorical(
        results_df['Period'], categories=['Pre-COVID', 'COVID', 'Crisis', 'Recovery'], ordered=True)
    results_df = results_df.sort_values(['Asset', 'Period']).reset_index(drop=True)

    print("\n--- Analysis Complete ---")
    n_bad = int((~results_df['Plausible Region']).sum())
    print(f"Cells outside the plausible parameter region after multi-start: {n_bad}")
    print(results_df[['Period', 'Asset', 'Alpha (Shock)', 'Beta (Persistence)',
                      'Gamma (Leverage)', 'Gamma P-Value']].to_string())

    output_filename = 'egarch_1_1_results.csv'
    results_df.to_csv(output_filename, index=False)
    print(f"\nFull results successfully saved to {output_filename}")


if __name__ == "__main__":
    main()


Loading data...
Calculating log returns...
Fitting EGARCH(1,1) with 9 starts per cell...


C:\Users\sabit\AppData\Local\Temp\ipykernel_15752\4041564716.py:43: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  candidates.append(am.fit(disp='off', starting_values=sv,


C:\Users\sabit\AppData\Local\Temp\ipykernel_15752\4041564716.py:33: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  candidates.append(am.fit(disp='off', options={'maxiter': 2000}))



--- Analysis Complete ---
Cells outside the plausible parameter region after multi-start: 0
       Period                                   Asset  Alpha (Shock)  Beta (Persistence)  Gamma (Leverage)  Gamma P-Value
0   Pre-COVID                                    ASPI       0.154815        9.178986e-01          0.028288   4.581399e-01
1       COVID                                    ASPI       0.476252        9.364772e-01          0.006133   8.900329e-01
2      Crisis                                    ASPI       0.481481        6.467992e-01         -0.221478   4.725987e-03
3    Recovery                                    ASPI       0.402861        8.480567e-01         -0.078470   6.910209e-02
4   Pre-COVID                                   Banks       0.482281        8.162712e-01          0.116562   1.012336e-01
5       COVID                                   Banks       0.415676        9.427001e-01         -0.024335   6.446554e-01
6      Crisis                                   Banks